In [1]:
import os
import sys
from pathlib import Path

# Start from the current notebook directory
current_path = Path(os.getcwd()).resolve()

# Climb up until we find the directory containing 'requirements.txt'
project_root = None
for parent in [current_path] + list(current_path.parents):
    if (parent / "requirements.txt").exists():
        project_root = parent
        break

# If found, set it as working directory and update sys.path
if project_root:
    os.chdir(project_root)
    if str(project_root) not in sys.path:
        sys.path.append(str(project_root))
    print(f"Current Working Directory set to root: {project_root}")
else:
    print("Error: Could not find project root directory automatically.")

Current Working Directory set to root: C:\Users\alefi\OneDrive\Documents\OneDrive\Documents\Courses\Complete_MLOps_Course_KN\end_to_end_DS_proj


In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path
    

In [3]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml,create_directories
class ConfigurationManager:
    def __init__(self,
                config_filepath=CONFIG_FILE_PATH,
                params_filepath=PARAMS_FILE_PATH,
                schema_filepath=SCHEMA_FILE_PATH):
        self.config=read_yaml(config_filepath)
        self.params=read_yaml(params_filepath)
        self.schema=read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self)-> DataIngestionConfig:
        config=self.config.data_ingestion
        create_directories([config.root_dir])

        data_ingestion_config=DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )
        return data_ingestion_config

In [6]:
import os
import urllib.request as request
import zipfile
import ssl 
from src.datascience import logger
from src.datascience.entity.config_entity import DataIngestionConfig

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            logger.info("Attempting to download dataset...")
            
            # Set the unverified context globally for urllib in this session
            ssl._create_default_https_context = ssl._create_unverified_context
            
            
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f"{filename} download successfully with the following info:\n{headers}")
        else:
            logger.info(f"File already exists at: {self.config.local_data_file}")

    def extract_zip_file(self):
        """
        Extracts the zip file into the data ingestion directory
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)
        logger.info(f"Zip file extracted successfully to: {unzip_path}")

In [7]:
try:
    config=ConfigurationManager()
    data_ingestion_config=config.get_data_ingestion_config()
    data_ingestion=DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-06-11 14:25:18,536: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-11 14:25:18,547: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-11 14:25:18,549: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-11 14:25:18,551: INFO: common: created directory at: artifacts]
[2026-06-11 14:25:18,557: INFO: common: created directory at: artifacts/data_ingestion]
[2026-06-11 14:25:18,561: INFO: 4074487827: Attempting to download dataset...]


[2026-06-11 14:25:19,421: INFO: 4074487827: artifacts/data_ingestion/data.zip download successfully with the following info:
Connection: close
Content-Length: 23329
Cache-Control: max-age=300
Content-Security-Policy: default-src 'none'; style-src 'unsafe-inline'; sandbox
Content-Type: application/zip
ETag: "c69888a4ae59bc5a893392785a938ccd4937981c06ba8a9d6a21aa52b4ab5b6e"
Strict-Transport-Security: max-age=31536000
X-Content-Type-Options: nosniff
X-Frame-Options: deny
X-XSS-Protection: 1; mode=block
X-GitHub-Request-Id: 1B44:1A5442:118299:2082B7:6A2A77F7
Accept-Ranges: bytes
Date: Thu, 11 Jun 2026 08:55:19 GMT
Via: 1.1 varnish
X-Served-By: cache-maa10246-MAA
X-Cache: MISS
X-Cache-Hits: 0
X-Timer: S1781168119.213512,VS0,VE296
Vary: Authorization,Accept-Encoding
Access-Control-Allow-Origin: *
Cross-Origin-Resource-Policy: cross-origin
X-Fastly-Request-ID: b7f14b7ea9cbbc8d3bd419ed09b010f45177598e
Expires: Thu, 11 Jun 2026 09:00:19 GMT
Source-Age: 0

]
[2026-06-11 14:25:19,463: INFO: 40744